# 14. Trace Store — Hash-chained audit

Every LLM call leaves a fingerprint. When a federal auditor asks "prove this conversation happened, exactly as you say it did, six months ago," you need an answer that doesn't depend on trust. This notebook builds that answer.

`TraceStore` is arcllm's append-only audit log for LLM invocations. Every record links to the previous one via SHA-256 — change one byte and the chain breaks. Records serialize through RFC 8785 JCS (JSON Canonicalization Scheme) so the same data always hashes to the same value, regardless of dict ordering or whitespace. Files rotate daily and are owner-read-only (`chmod 0o600`). On warm start, the store re-verifies the tail of the chain and refuses to stay silent if it's been touched.

**You will learn:**
- Why hash chains satisfy NIST AU-10 (Non-repudiation) and CMMC AU.L2-3.3.1 (Audit retention)
- The `TraceRecord` schema — what's recorded, what's hashed, what's optional
- `JSONLTraceStore` — daily rotation, hash chaining, file permissions
- RFC 8785 JCS canonical JSON — why the same record always hashes identically
- Tamper detection on warm start — mutating a record on disk and watching it fail
- Cursor pagination, filtering by provider/agent/status, streaming with `iter_records`
- Threading the store into `load_model(trace_store=...)` for end-to-end audit


## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

We bring in only what we need — the `TraceRecord` schema, the `JSONLTraceStore` implementation, and a few helpers. Notice that `TraceStore` is a `Protocol`: any object with `append`, `query`, `get`, `verify_chain`, `iter_records`, and `close` satisfies the contract. JSONLTraceStore is the only built-in concrete impl, but the rest of arcllm only depends on the Protocol.

In [ ]:
import asyncio
import hashlib
import json
import logging
import tempfile
from datetime import UTC, datetime
from pathlib import Path

from arcllm.trace_store import JSONLTraceStore, TraceRecord, TraceStore

# A tmp_path stand-in for notebook cells. Keeping it close to module-level
# so later cells can recreate fresh stores under SANDBOX/ subdirs.
SANDBOX = Path(tempfile.mkdtemp(prefix="arcllm-trace-walkthrough-"))
print(f"sandbox: {SANDBOX}")
print(f"TraceStore is a Protocol: {TraceStore.__module__}.{TraceStore.__name__}")

## 2. Why hash chains

A plain JSONL log file is *append-only by convention* — anyone with a text editor can rewrite history. Hash chains turn that convention into math. Each record stores the SHA-256 of the previous record (`prev_hash`) plus its own hash (`record_hash`) computed over its own contents. Mutating record N invalidates record N's `record_hash`; even if you recompute that, record N+1 still points at the *old* hash through `prev_hash`.

**What that buys us:**

| Property | Guarantee |
|---|---|
| Tamper-evident | Any byte mutation in any record breaks `verify_chain()` |
| Order-evident | You can't reorder records — hashes link them |
| Insert-evident | You can't slip a record in — neighbors hash-disagree |
| Delete-evident | You can't remove a record — chain ends prematurely |

**Compliance map:**

- **NIST 800-53 AU-10** — Non-repudiation. Cryptographic linkage prevents agents from claiming "I didn't say that."
- **NIST 800-53 AU-9** — Protection of audit information. We chmod traces to `0o600` and store them outside the agent's tool sandbox.
- **CMMC AU.L2-3.3.1** — Create and retain audit logs.
- **OWASP ASI-10** — Rogue agents. Behavioural anomalies surface in the trace; tamper with the trace and warm-start refuses to stay quiet.

In [ ]:
# A 30-second visual proof: what a 1-byte mutation does to a sha256.
a = b'{"trace_id": "abc", "cost_usd": 0.001}'
b = b'{"trace_id": "abc", "cost_usd": 0.002}'  # one digit changed
print("hash(a):", hashlib.sha256(a).hexdigest())
print("hash(b):", hashlib.sha256(b).hexdigest())
print("identical?", hashlib.sha256(a).digest() == hashlib.sha256(b).digest())

Two records that differ in exactly one digit produce hashes that share zero bytes. That avalanche property is what we exploit: an auditor only needs to recompute the chain to know whether the file was touched.

## 3. `TraceRecord` schema

`TraceRecord` is a frozen Pydantic model. Frozen because audit records can't be edited after they're written — the type system enforces this.

**Field families:**

| Family | Fields | Notes |
|---|---|---|
| Identity | `trace_id`, `timestamp`, `agent_label`, `agent_did`, `budget_scope` | UUID4 hex by default; ISO-8601 UTC |
| Provider | `provider`, `model` | Keyed to TOML config in `arcllm/config/providers/` |
| Bodies | `request_body`, `response_body` | Optional — federal tier stores; personal may not |
| Telemetry | `duration_ms`, `cost_usd`, `input_tokens`, `output_tokens`, `cache_*_tokens`, `total_tokens`, `stop_reason` | Always populated for `event_type="llm_call"` |
| Outcome | `status` (`success`/`error`/`timeout`), `error` | |
| Retry | `attempt_number`, `retry_group_id` | Links a chain of attempts |
| Phases | `phase_timings` | `prompt_assembly_ms`, `llm_call_ms`, `post_processing_ms`, etc. |
| Discriminator | `event_type`, `event_data` | `llm_call`, `config_change`, `circuit_change`, `rotation` |
| Chain | `prev_hash`, `record_hash` | The cryptographic linkage |

Every field except `record_hash` participates in the hash. We exclude `record_hash` for the obvious reason: you can't include a value in the input to its own computation.

In [ ]:
# A minimal valid record — only `provider` and `model` are required.
rec = TraceRecord(provider="anthropic", model="claude-sonnet-4-6")

print("trace_id:    ", rec.trace_id, "  (uuid4 hex, len 32)")
print("timestamp:   ", rec.timestamp, "  (ISO-8601 UTC)")
print("event_type:  ", rec.event_type)
print("status:      ", rec.status)
print("prev_hash:   ", rec.prev_hash[:16] + "...", "  (genesis = 64 zeros)")
print("record_hash: ", repr(rec.record_hash), "  (empty until with_hash() is called)")

In [ ]:
# A richer record with telemetry, phases, and a request/response body.
rich = TraceRecord(
    provider="anthropic",
    model="claude-sonnet-4-6",
    agent_label="scap_isso",
    agent_did="did:arc:scap_isso",
    budget_scope="agent:scap_isso",
    request_body={"messages": [{"role": "user", "content": "hi"}]},
    response_body={"content": "hello", "stop_reason": "end_turn"},
    duration_ms=812.4,
    cost_usd=0.00123,
    input_tokens=120,
    output_tokens=45,
    total_tokens=165,
    cache_read_tokens=80,
    phase_timings={
        "prompt_assembly_ms": 12.1,
        "llm_call_ms": 798.0,
        "post_processing_ms": 2.3,
        "total_ms": 812.4,
    },
    stop_reason="end_turn",
)
print("agent_label =", rich.agent_label)
print("phase keys  =", sorted(rich.phase_timings.keys()))
print("cost_usd    =", rich.cost_usd)

TraceRecord is `frozen=True` — once created, it can't be mutated. This is the type system protecting us from accidentally rewriting history in code.

In [ ]:
from pydantic import ValidationError

try:
    rich.cost_usd = 999.99  # type: ignore[misc]
except ValidationError as e:
    print("frozen → ValidationError raised:", type(e).__name__)
    print("  field            =", e.errors()[0]["loc"])
    print("  message contains =", e.errors()[0]["msg"])

**Three event types beyond `llm_call`** ride the same chain — that's how config changes, circuit-breaker transitions, and daily rotations get audit-trailed alongside the calls themselves.

In [ ]:
config_change = TraceRecord(
    provider="system",
    model="system",
    event_type="config_change",
    event_data={
        "actor": "operator",
        "changes": {"temperature": {"old": 0.7, "new": 0.5}},
    },
)
circuit_change = TraceRecord(
    provider="anthropic",
    model="claude-sonnet-4-6",
    event_type="circuit_change",
    event_data={"old_state": "closed", "new_state": "open", "reason": "5 consecutive failures"},
)
print(config_change.event_type, "→", config_change.event_data["changes"])
print(circuit_change.event_type, "→", circuit_change.event_data)

## 4. `JSONLTraceStore` basics

`JSONLTraceStore` is the file-backed implementation. Pass it an `agent_root` (a Path) and it'll create `<agent_root>/traces/` (with `chmod 0o700`) and write daily files named `traces-YYYY-MM-DD.jsonl`. Files inside are `chmod 0o600` — owner-read-only.

Why the `traces/` subdirectory lives at the agent root and not inside `workspace/`: **NIST AU-9**. The agent's tool sandbox can read and write inside `workspace/` to do its job; if the audit log lived there, a compromised tool could rewrite history. Putting traces *outside* the sandbox enforces a sharper trust boundary.

Every `append()` call is async, lock-protected, and computes the hash chain automatically — caller never touches `prev_hash` or `record_hash` directly.

In [ ]:
# Create a fresh store under a sandbox directory.
agent_root = SANDBOX / "agent_alpha"
store = JSONLTraceStore(agent_root)

print("traces dir:", store._traces_dir)
print("exists?   ", store._traces_dir.exists())
print("perms     ", oct(store._traces_dir.stat().st_mode & 0o777), "  (0o700 = owner rwx)")

In [ ]:
# Append three records. Each has a distinct trace_id; everything else is shared.
async def write_three():
    for i in range(3):
        await store.append(
            TraceRecord(
                trace_id=f"alpha-{i:03d}",
                provider="anthropic",
                model="claude-sonnet-4-6",
                agent_label="alpha",
                input_tokens=100 + i * 10,
                output_tokens=50,
                total_tokens=150 + i * 10,
                cost_usd=0.001 * (i + 1),
            )
        )


await write_three()
today = datetime.now(UTC).strftime("%Y-%m-%d")
log_file = agent_root / "traces" / f"traces-{today}.jsonl"
print("file:    ", log_file)
print("perms:   ", oct(log_file.stat().st_mode & 0o777), "  (0o600 = owner rw)")
print("line cnt:", len(log_file.read_text().splitlines()))

Let's look at the file on disk. Each line is a complete JSON record — JSONL means "newline-delimited JSON," the format every log aggregator already understands.

In [ ]:
raw = log_file.read_text().splitlines()
for i, line in enumerate(raw):
    rec = json.loads(line)
    print(f"line {i}: trace_id={rec['trace_id']}")
    print(f"        prev_hash    = {rec['prev_hash'][:32]}...")
    print(f"        record_hash  = {rec['record_hash'][:32]}...")
    print(f"        cost_usd     = {rec['cost_usd']}")

Notice the chain: line 0's `prev_hash` is 64 zeros (the genesis marker), and each subsequent line's `prev_hash` matches the previous line's `record_hash`. Let's check that with code rather than eyeballing it.

In [ ]:
records = [json.loads(l) for l in raw]
GENESIS = "0" * 64

assert records[0]["prev_hash"] == GENESIS, "first record must point at genesis"
for i in range(1, len(records)):
    assert records[i]["prev_hash"] == records[i - 1]["record_hash"], f"chain break at index {i}"
print("chain intact across", len(records), "records")

In [ ]:
# verify_chain() does the same check, plus re-derives every record_hash to confirm
# nothing inside any record was edited.
ok = await store.verify_chain()
print("verify_chain():", ok)

## 5. RFC 8785 JCS — canonical JSON

Plain `json.dumps()` is *not* deterministic. Different Python versions, different dict insertion orders, and different whitespace settings can all produce equivalent JSON that hashes differently. That's a non-starter for an audit chain — re-serializing a record to verify it would routinely fail not because of tampering, but because the bytes drifted.

**RFC 8785** (the JSON Canonicalization Scheme, "JCS") fixes this:

- Object keys are sorted lexicographically (UTF-16 code-unit order).
- Numbers use the shortest round-trip representation per ECMA-404.
- No insignificant whitespace.
- Strings escape exactly the characters JSON requires, no more.

`TraceRecord.compute_hash()` calls `jcs.canonicalize()` before hashing. Same data → same canonical bytes → same hash, regardless of how the dict was built.

Let's prove it.

In [ ]:
import jcs  # type: ignore[import-untyped]

# Same payload, three different orderings.
payload_a = {"provider": "anthropic", "model": "claude-sonnet-4-6", "cost_usd": 0.001}
payload_b = {"cost_usd": 0.001, "model": "claude-sonnet-4-6", "provider": "anthropic"}
payload_c = {"model": "claude-sonnet-4-6", "cost_usd": 0.001, "provider": "anthropic"}

# Plain json.dumps preserves insertion order — three different byte strings.
print("json.dumps (NOT canonical):")
for name, p in [("a", payload_a), ("b", payload_b), ("c", payload_c)]:
    print(f"  {name}: {json.dumps(p)}")

# jcs.canonicalize sorts keys → identical bytes every time.
print("\njcs.canonicalize (canonical):")
for name, p in [("a", payload_a), ("b", payload_b), ("c", payload_c)]:
    print(f"  {name}: {jcs.canonicalize(p).decode()}")

In [ ]:
# And of course, identical canonical bytes → identical SHA-256 hashes.
h_a = hashlib.sha256(jcs.canonicalize(payload_a)).hexdigest()
h_b = hashlib.sha256(jcs.canonicalize(payload_b)).hexdigest()
h_c = hashlib.sha256(jcs.canonicalize(payload_c)).hexdigest()
print("hash(a):", h_a)
print("hash(b):", h_b)
print("hash(c):", h_c)
print("all equal?", h_a == h_b == h_c)

In [ ]:
# TraceRecord.compute_hash() encapsulates the same dance.
rec_a = TraceRecord(
    trace_id="jcs-demo",
    timestamp="2026-05-07T00:00:00+00:00",
    provider="anthropic",
    model="claude-sonnet-4-6",
    cost_usd=0.001,
    input_tokens=100,
    output_tokens=50,
    total_tokens=150,
)
rec_b = rec_a.model_copy()  # exactly the same data, possibly different dict ordering

print("compute_hash(a):", rec_a.compute_hash())
print("compute_hash(b):", rec_b.compute_hash())
print("deterministic?  ", rec_a.compute_hash() == rec_b.compute_hash())

Two takeaways: (1) the hash is reproducible months later, by anyone with the same data; (2) we can serialize records however we like (Pydantic's `model_dump_json()`, manual `json.dumps`, anything) and the hash check still works because we always re-canonicalize before hashing.

## 6. Daily rotation

JSONL files would grow without bound if we kept writing to one per agent forever. `JSONLTraceStore._maybe_rotate()` checks the current UTC date on every `append()` and rotates to a new file at the day boundary. The trick: the chain has to survive the rotation. The store does this by writing a special **rotation tombstone** as the last record of the closing file. The tombstone has `event_type="rotation"`, `event_data={"next_file": "traces-YYYY-MM-DD.jsonl"}`, and its `record_hash` becomes the `prev_hash` of the first real record in the new file.

Tombstones are skipped during normal queries (`query()` filters them out) so applications never see them. They're a chain artifact, not a user-facing event.

We can simulate a rotation by patching the date function.

In [ ]:
from unittest.mock import patch

rotation_root = SANDBOX / "agent_rotating"
rotation_store = JSONLTraceStore(rotation_root)

# Day 1: write two records.
with patch.object(rotation_store, "_today", return_value="2026-05-06"):
    await rotation_store.append(
        TraceRecord(trace_id="day1-a", provider="anthropic", model="claude-sonnet-4-6")
    )
    await rotation_store.append(
        TraceRecord(trace_id="day1-b", provider="anthropic", model="claude-sonnet-4-6")
    )

# Day 2: write one record. The first append on day 2 closes day 1 with a tombstone.
with patch.object(rotation_store, "_today", return_value="2026-05-07"):
    await rotation_store.append(
        TraceRecord(trace_id="day2-a", provider="anthropic", model="claude-sonnet-4-6")
    )

files = sorted((rotation_root / "traces").glob("traces-*.jsonl"))
for f in files:
    lines = f.read_text().splitlines()
    print(f"{f.name}: {len(lines)} lines")
    for line in lines:
        rec = json.loads(line)
        print(f"   trace_id={rec['trace_id']:<12} event_type={rec['event_type']}")

Day 1's file has 3 lines: 2 real records plus the rotation tombstone. Day 2's file has the new record. The tombstone's `record_hash` is the `prev_hash` of `day2-a` — the chain spans the rotation.

In [ ]:
# Verify the chain crosses the file boundary.
day1 = json.loads(
    (rotation_root / "traces" / "traces-2026-05-06.jsonl").read_text().splitlines()[-1]
)
day2 = json.loads(
    (rotation_root / "traces" / "traces-2026-05-07.jsonl").read_text().splitlines()[0]
)

print("day1 last (tombstone):")
print(f"  event_type   = {day1['event_type']}")
print(f"  next_file    = {day1['event_data']['next_file']}")
print(f"  record_hash  = {day1['record_hash'][:32]}...")
print("day2 first:")
print(f"  trace_id     = {day2['trace_id']}")
print(f"  prev_hash    = {day2['prev_hash'][:32]}...")
print("chain crosses files?", day1["record_hash"] == day2["prev_hash"])

In [ ]:
# verify_chain() walks every file in date order — chain must hold across all of them.
ok = await rotation_store.verify_chain()
print("verify_chain across rotation:", ok)

## 7. Tamper detection — the demo that proves the point

Theory is one thing; let's actually break a record and watch the system catch it.

We'll write five records, close the store, mutate one byte of the third record on disk, reopen the store, and observe two things:

1. **`_warm_start()` logs `TAMPER DETECTED`** — the warm-start tail check (last 10 records) logs at ERROR level when it detects a chain break or hash mismatch.
2. **`verify_chain()` returns `False`** — full-chain verification refuses to lie about it.

In [ ]:
# Set up: write five records.
tamper_root = SANDBOX / "agent_tampered"
tamper_store = JSONLTraceStore(tamper_root)
for i in range(5):
    await tamper_store.append(
        TraceRecord(
            trace_id=f"clean-{i:03d}",
            provider="anthropic",
            model="claude-sonnet-4-6",
            cost_usd=0.001 * (i + 1),
        )
    )
await tamper_store.close()

today = datetime.now(UTC).strftime("%Y-%m-%d")
tamper_file = tamper_root / "traces" / f"traces-{today}.jsonl"
lines = tamper_file.read_text().splitlines()
print(f"clean state: {len(lines)} records, chain valid =", await tamper_store.verify_chain())

In [ ]:
# Tamper: change cost_usd in record index 2 (the middle one) without recomputing hashes.
before = json.loads(lines[2])
before_cost = before["cost_usd"]
before["cost_usd"] = 99999.99
lines[2] = json.dumps(before)
tamper_file.chmod(0o600)
tamper_file.write_text("\n".join(lines) + "\n")
print(f"record 2 cost_usd: {before_cost} → 99999.99 (tampered)")
print("  the record_hash field is now stale relative to the data")

In [ ]:
# Reopen the store. _warm_start() runs the tail check on the next append.
# We capture the logger to see the TAMPER DETECTED message.
tamper_logger = logging.getLogger("arcllm.trace_store")
captured: list[logging.LogRecord] = []


class CaptureHandler(logging.Handler):
    def emit(self, rec: logging.LogRecord) -> None:
        captured.append(rec)


handler = CaptureHandler(level=logging.ERROR)
tamper_logger.addHandler(handler)
tamper_logger.setLevel(logging.ERROR)

reopened = JSONLTraceStore(tamper_root)
# Trigger warm_start by calling append. Use a no-op record we won't actually need.
await reopened.append(
    TraceRecord(trace_id="after-tamper", provider="anthropic", model="claude-sonnet-4-6")
)
tamper_logger.removeHandler(handler)

for rec in captured:
    print(f"[{rec.levelname}] {rec.getMessage()}")
if not captured:
    print("(no errors logged — tail check did not flag this; falling back to full verify)")

In [ ]:
# verify_chain() is the ground-truth check: walks every record, recomputes every hash.
ok = await reopened.verify_chain()
if ok:
    print("chain valid — tamper NOT detected (this is a bug if it shows up)")
else:
    # Find the first broken link by walking the file by hand.
    raw = tamper_file.read_text().splitlines()
    prev = "0" * 64
    for idx, line in enumerate(raw):
        data = json.loads(line)
        rec = TraceRecord(**data)
        broken_link = rec.prev_hash != prev
        broken_self = rec.record_hash != rec.compute_hash()
        if broken_link or broken_self:
            reason = "prev_hash mismatch" if broken_link else "record_hash mismatch"
            print(f"tamper detected at record {idx}: {reason}")
            print(f"  trace_id   = {rec.trace_id}")
            print(
                f"  expected   = {prev[:16]}..."
                if broken_link
                else f"  recomputed = {rec.compute_hash()[:16]}..."
            )
            print(
                f"  found      = {rec.prev_hash[:16]}..."
                if broken_link
                else f"  stored     = {rec.record_hash[:16]}..."
            )
            break
        prev = rec.record_hash
    else:
        print("verify_chain returned False but walk found nothing — investigate")

**That's the whole point.** A single byte changed five hours, five days, or five months after the fact still surfaces. The store can't be silently rewritten. If an attacker wanted to hide a record, they'd have to recompute every hash from the tampered point forward — which they can do, but if any external system already mirrored a hash from the original chain (a witness server, an OTel exporter, an immutable storage tier), the rewritten chain won't match the witness.

## 8. Pagination & queries

`query()` reads records *newest-first* with cursor pagination, optional filters (`provider`, `agent`, `status`, `start`, `end`), and a configurable `limit` (default 50). It also skips rotation tombstones automatically.

Internally, it uses a chunked reverse line reader so a `limit=50` query against a 10K-line file reads ~50 records, not 10K. That's the difference between an API call that takes milliseconds and one that takes seconds — federations of stores amplify the win.

Three useful query patterns:

In [ ]:
# Set up: a fresh store with a mix of providers, agents, and statuses.
query_root = SANDBOX / "agent_queries"
query_store = JSONLTraceStore(query_root)

samples = [
    ("anthropic", "alpha", "success"),
    ("openai", "alpha", "success"),
    ("anthropic", "beta", "error"),
    ("anthropic", "alpha", "success"),
    ("openai", "beta", "timeout"),
    ("anthropic", "alpha/memory", "success"),  # sub-agent label
    ("google", "beta", "success"),
    ("anthropic", "alpha", "error"),
]
for i, (prov, agent, status) in enumerate(samples):
    await query_store.append(
        TraceRecord(
            trace_id=f"q-{i:03d}",
            timestamp=f"2026-05-07T00:00:{i:02d}+00:00",
            provider=prov,
            model="x",
            agent_label=agent,
            status=status,  # type: ignore[arg-type]
        )
    )
print(f"wrote {len(samples)} records")

In [ ]:
# Pattern 1: page through with limit + cursor.
page1, cursor1 = await query_store.query(limit=3)
print("page 1 (newest first):")
for r in page1:
    print(f"  {r.trace_id} | {r.provider:<10} | {r.agent_label:<14} | {r.status}")
print(f"cursor1: {cursor1}")

page2, cursor2 = await query_store.query(limit=3, cursor=cursor1)
print("\npage 2:")
for r in page2:
    print(f"  {r.trace_id} | {r.provider:<10} | {r.agent_label:<14} | {r.status}")
print(f"cursor2: {cursor2}")

In [ ]:
# Pattern 2: filter by provider, agent, status.
anthropic_errors, _ = await query_store.query(provider="anthropic", status="error")
print("anthropic errors:", [r.trace_id for r in anthropic_errors])

alpha_records, _ = await query_store.query(agent="alpha")
# Note: agent='alpha' prefix-matches 'alpha/memory' too — sub-agent semantics.
print("agent=alpha (prefix-matches alpha/memory):", [r.agent_label for r in alpha_records])

successes, _ = await query_store.query(status="success")
print("status=success count:", len(successes))

In [ ]:
# Pattern 3: time-window filter via ISO-8601 strings.
early, _ = await query_store.query(
    start="2026-05-07T00:00:00+00:00",
    end="2026-05-07T00:00:03+00:00",
)
print("first 4 seconds (inclusive):", [r.trace_id for r in early])

In [ ]:
# Pattern 4: get a specific record by trace_id.
found = await query_store.get("q-005")
missing = await query_store.get("does-not-exist")
print(
    "get('q-005')           :",
    None if found is None else f"{found.trace_id} agent={found.agent_label}",
)
print("get('does-not-exist'): ", missing)

In [ ]:
# Pattern 5: stream every record without loading any single file fully into memory.
# iter_records() yields plain dicts in chronological storage order — useful for federation
# (multi-agent merge), backfill, and warm-start replay (SPEC-019 T2).
all_seen = []
async for record_dict in query_store.iter_records():
    all_seen.append(record_dict["trace_id"])
print(f"iter_records yielded {len(all_seen)} dicts:", all_seen)

Two subtle behaviours to call out:

- **Agent prefix matching.** `agent="alpha"` matches both `"alpha"` and `"alpha/memory"`. Sub-components of an agent (like its memory subsystem) emit traces under `parent/sub` labels, and the API treats them as part of the parent's traces. Exact match would silently hide every sub-component event from the dashboard.
- **Tombstones are invisible.** `query()` and `iter_records()` both skip rotation events. If you want to see them, you can read the JSONL files directly.

## 9. Integration with `load_model()`

`TraceStore` doesn't get wired into the module stack manually — `load_model()` accepts `trace_store=`, `on_event=`, and `agent_label=` kwargs and threads them into the `TelemetryModule` config. The module computes phase timings, builds the `TraceRecord`, fires `on_event(record)` (if provided), and calls `await store.append(record)` (if provided). They're independent — set either, both, or neither.

**Stack position recap:** `Otel → Queue → Telemetry → CircuitBreaker → Audit → Security → Retry → Fallback → RateLimit → [Adapter|Router]`. `Telemetry` sits high enough to see total wall-clock duration through retries and rate-limit waits, but inside `Queue`/`Otel` so queue-wait time isn't double-counted in `phase_timings`.

We'll demonstrate by wiring up `TelemetryModule` directly with a mock adapter — exactly the shape that `load_model()` constructs internally — so we can run end-to-end without an API key.

In [ ]:
from unittest.mock import AsyncMock, MagicMock

from arcllm.modules.telemetry import TelemetryModule
from arcllm.types import LLMProvider, LLMResponse, Message, Usage

integration_root = SANDBOX / "agent_integration"
integration_store = JSONLTraceStore(integration_root)
captured_events: list[TraceRecord] = []

# Mock adapter — three canned responses.
responses = [
    LLMResponse(
        content="first",
        usage=Usage(input_tokens=100, output_tokens=50, total_tokens=150),
        model="claude-sonnet-4-6",
        stop_reason="end_turn",
    ),
    LLMResponse(
        content="second",
        usage=Usage(input_tokens=200, output_tokens=80, total_tokens=280, cache_read_tokens=50),
        model="claude-sonnet-4-6",
        stop_reason="end_turn",
    ),
    LLMResponse(
        content="third",
        usage=Usage(input_tokens=300, output_tokens=120, total_tokens=420),
        model="claude-sonnet-4-6",
        stop_reason="max_tokens",
    ),
]
inner = MagicMock(spec=LLMProvider)
inner.name = "anthropic"
inner.model_name = "claude-sonnet-4-6"
inner.invoke = AsyncMock(side_effect=responses)

# This is exactly the config dict load_model() assembles for telemetry.
config = {
    "cost_input_per_1m": 3.00,
    "cost_output_per_1m": 15.00,
    "cost_cache_read_per_1m": 0.30,
    "cost_cache_write_per_1m": 3.75,
    "trace_store": integration_store,
    "on_event": captured_events.append,
    "agent_label": "scap_isso",
}
module = TelemetryModule(config, inner)
print("telemetry module wired with trace_store + on_event + agent_label")

In [ ]:
# Drive three calls.
for i in range(3):
    await module.invoke([Message(role="user", content=f"call {i}")])

# on_event fired for every call.
print(f"on_event captured {len(captured_events)} records")
for rec in captured_events:
    print(
        f"  {rec.trace_id[:8]}... | {rec.provider} | tokens={rec.input_tokens}/{rec.output_tokens}"
        f" | cost=${rec.cost_usd:.6f} | agent={rec.agent_label}"
    )

In [ ]:
# Phase timings on each record — built by TelemetryModule, not the store.
for rec in captured_events:
    print(f"{rec.trace_id[:8]}... phases:")
    for phase, ms in rec.phase_timings.items():
        print(f"  {phase:<24} {ms:.3f} ms")

In [ ]:
# And the JSONL store has the same three records, hash-chained.
stored, _ = await integration_store.query(limit=10)
stored.reverse()  # query is newest-first; reverse for chronological
print(f"trace_store has {len(stored)} records")
for rec in stored:
    print(f"  {rec.trace_id[:8]}... agent={rec.agent_label} status={rec.status}")

print("chain valid?", await integration_store.verify_chain())

**That's the production wiring.** In real code:

```python
from arcllm import load_model
from arcllm.trace_store import JSONLTraceStore

store = JSONLTraceStore(Path("/var/arc/agents/scap_isso"))
model = load_model(
    "anthropic",
    trace_store=store,
    on_event=lambda r: ws.broadcast(r.model_dump()),  # live UI
    agent_label="scap_isso",
    telemetry=True,
)
async with model:
    response = await model.invoke(messages)
```

Every call automatically lands in the JSONL trace, on disk, hash-chained, with phase timings, cost, and agent attribution. No bookkeeping in your loop code.

## 10. Summary

**What we covered:**

- **Why hash chains exist.** Tamper-evident, order-evident, insert-evident, delete-evident. Maps to NIST AU-9, AU-10, CMMC AU.L2-3.3.1, OWASP ASI-10.
- **`TraceRecord`.** Frozen Pydantic model. UUID4 default `trace_id`, ISO-8601 default `timestamp`, four `event_type` discriminators (`llm_call`, `config_change`, `circuit_change`, `rotation`). Hash covers every field except `record_hash`.
- **`JSONLTraceStore`.** Append-only, daily-rotating, lock-protected, `0o600` audit files outside the agent's tool sandbox. Implements the `TraceStore` Protocol.
- **RFC 8785 JCS.** Same data → same canonical bytes → same SHA-256 → same chain link, across processes, languages, and Python versions.
- **Daily rotation + tombstones.** The chain crosses file boundaries; tombstones carry the hash forward and are invisible to `query()` / `iter_records()`.
- **Tamper detection.** Mutating one byte breaks `verify_chain()` and triggers warm-start ERROR logging the next time the store is opened.
- **Pagination & queries.** Cursor-based, newest-first, with provider/agent/status/time filters. Agent prefix-match for sub-component traces. `iter_records()` for streaming warm-start replay.
- **`load_model()` integration.** Pass `trace_store=`, `on_event=`, `agent_label=` kwargs; the `TelemetryModule` does the rest.

**Public API exercised:** `TraceStore` (Protocol), `JSONLTraceStore`, `TraceRecord` — all three from `arcllm.trace_store`. Plus the `trace_store=`, `on_event=`, `agent_label=` kwargs on `load_model()` (via the `TelemetryModule` config dict that `load_model()` assembles).

**Where to next:**

- Notebook **10 — Audit trail** for the `arctrust.audit.emit` sinks side of the story (`JsonlSink`, `SignedChainSink`, custom sinks).
- Notebook **09 — Telemetry module** for cost calculation and `on_event` semantics.
- Notebook **16 — Config controller** for how runtime config mutations land as `event_type="config_change"` records on the same chain.
- arctrust notebook **04 — Audit sinks** for the broader sink architecture and how TraceStore fits alongside `arctrust.audit`.